# SCOPE: uncorrected vs corrected $w_p$ — 2×2 convergence grid

Four-panel layout:

| | Naïve (no correction) | SCOPE corrected |
|---|---|---|
| **$\log_{10}w_p$** | top-left | top-right |
| **$\Delta\log_{10}w_p$** | bottom-left | bottom-right |

Each curve is the mean over all seeds; coloured by $n_{\rm subvol}$ on a rainbow scale.

**Data source:** CSVs from `scripts/2pcf/slurm/submit_scope_wp_l800_lc16_campaign.sh`.

In [ ]:
import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('../../src').resolve()))
from galform_analysis.utils.matplotlib_config import setconfig
from galform_analysis.config import get_snapshot_redshift

setconfig()

In [ ]:
DATA_ROOT = Path('../../data/2pcf/scope_wp')

MODEL     = 'lc16'
IZ        = 155
CENTRALS  = 1
MSTAR_TAG = 'mstar10.0'
MHALO_TAG = 'none'
N_REF     = 1024   # reference n_subvol

Z = get_snapshot_redshift(f'iz{IZ}')
print(f'iz{IZ}  →  z = {Z:.3f}')

## 1  Load and aggregate

In [ ]:
run_dir   = DATA_ROOT / MODEL / f'iz{IZ}' / f'centrals_{CENTRALS}' / MSTAR_TAG / MHALO_TAG
csv_files = sorted(run_dir.glob(f'seed*/scope_wp_*_iz{IZ}.csv'))

if not csv_files:
    raise FileNotFoundError(
        f'No CSVs under {run_dir}.\n'
        'Run:  bash scripts/2pcf/slurm/submit_scope_wp_l800_lc16_campaign.sh'
    )

raw = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

n_values = sorted(raw['n_subvol'].unique())
print(f'{len(csv_files)} files | {raw["selection_seed"].nunique()} seeds | n_subvol: {n_values}')

In [ ]:
# Mean over seeds for both corrected and naive
agg = (
    raw.groupby(['n_subvol', 'bin_idx', 'r_p'], as_index=False)
    .agg(
        wp_corr_mean  = ('wp_corrected', 'mean'),
        wp_naive_mean = ('wp_naive',     'mean'),
    )
)

# Reference log10 w_p from n=N_REF (corrected)
ref_corr  = (
    agg[agg['n_subvol'] == N_REF][['r_p', 'wp_corr_mean']]
    .rename(columns={'wp_corr_mean': 'wp_corr_ref'})
)
ref_naive = (
    agg[agg['n_subvol'] == N_REF][['r_p', 'wp_naive_mean']]
    .rename(columns={'wp_naive_mean': 'wp_naive_ref'})
)

agg = agg.merge(ref_corr,  on='r_p', how='left')
agg = agg.merge(ref_naive, on='r_p', how='left')

EPS = 1e-30
agg['log10_wp_corr']  = np.log10(agg['wp_corr_mean'].clip(lower=EPS))
agg['log10_wp_naive'] = np.log10(agg['wp_naive_mean'].clip(lower=EPS))

agg['dlog10_wp_corr']  = agg['log10_wp_corr']  - np.log10(agg['wp_corr_ref'].clip(lower=EPS))
agg['dlog10_wp_naive'] = agg['log10_wp_naive'] - np.log10(agg['wp_naive_ref'].clip(lower=EPS))

agg['log10_r_p'] = np.log10(agg['r_p'])

print(agg[['n_subvol', 'r_p', 'log10_wp_corr', 'dlog10_wp_corr']].head(8).to_string(index=False))

## 2  2×2 grid

In [ ]:
cmap   = mpl.colormaps['rainbow']
colors = cmap(np.linspace(0.0, 1.0, len(n_values)))
marker = 's'   # square markers, matching Image 2
ms     = 5

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True)
ax_tl, ax_tr, ax_bl, ax_br = axes[0, 0], axes[0, 1], axes[1, 0], axes[1, 1]

def plot_panel(ax, col, ref_col, ylabel, title, is_residual=False):
    for n, color in zip(n_values, colors):
        sub = agg[agg['n_subvol'] == n].sort_values('log10_r_p')
        lw  = 2.5 if n == N_REF else 1.6
        alpha = 1.0 if n == N_REF else 0.85
        ax.plot(
            sub['log10_r_p'], sub[col],
            marker=marker, ms=ms, lw=lw, alpha=alpha,
            color=color, label=f'$N_{{\\rm subvol}}={n}$',
        )
    if is_residual:
        ax.axhline(0, color='black', lw=1.5, ls='--', zorder=5)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=12)

plot_panel(ax_tl, 'log10_wp_naive', 'wp_naive_ref',
           r'$\log_{10}\,w_p$', 'Naïve (no correction)')
ax_tl.text(0.04, 0.05, 'naïve corr. func.',
           transform=ax_tl.transAxes, fontsize=12, va='bottom')

plot_panel(ax_tr, 'log10_wp_corr', 'wp_corr_ref',
           r'$\log_{10}\,w_p$', 'SCOPE corrected')
ax_tr.text(0.04, 0.05, 'weighted corr. func.',
           transform=ax_tr.transAxes, fontsize=12, va='bottom')
ax_tr.legend(
    loc='upper right', ncol=1, fontsize=9,
    bbox_to_anchor=(1.28, 1.0),
)

plot_panel(ax_bl, 'dlog10_wp_naive', 'wp_naive_ref',
           r'$\Delta\log_{10}\,w_p$', '', is_residual=True)

plot_panel(ax_br, 'dlog10_wp_corr', 'wp_corr_ref',
           r'$\Delta\log_{10}\,w_p$', '', is_residual=True)

for ax in axes[1]:
    ax.set_xlabel(r'$\log_{10}\,r_p$  [$h^{-1}$Mpc]')

fig.suptitle(
    f'$w_p$ convergence: naïve vs SCOPE corrected  –  L800/{MODEL}  $z={Z:.2f}$',
    fontsize=14,
)
plt.tight_layout()
plt.show()